# LSTM

Using the same preprocessed processed_data_V3

Unlike the RF (which flattens each clip into summary statistics) or the CNN (which looks at local temporal windows with filters), the LSTM processes frames one at a time and maintains a hidden state across the full sequence. This allows it to model long-range dependencies between frames.

In [1]:
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Bidirectional, Dense, Dropout, BatchNormalization
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from tensorflow.keras.optimizers import Adam

DATA_DIR = './data/Processed_ASL_Data'
RANDOM_SEED = 80
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

print('GPUs visible to TF:', tf.config.list_physical_devices('GPU'))

GPUs visible to TF: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


***Loading Data***

Cell loads the preprocessed data we have from the preprocessing step/jupyter notebook. The development/training set contains 18 participants used for cross-validation with our implemented KGroupFolds, while the last 3 participants are not included in the dev.npz, so we can be sure that the model is learning the actual sign and not learning from the participants' specific signing patterns. 

In [2]:
# load data 
dev = np.load(os.path.join(DATA_DIR, 'dev.npz'))
test = np.load(os.path.join(DATA_DIR, 'test.npz'))
folds = np.load(os.path.join(DATA_DIR, 'cv_folds.npz'))
classes = np.load(os.path.join(DATA_DIR, 'classes.npy'), allow_pickle=True)

X_dev, y_dev, groups_dev = dev['X'], dev['y'], dev['groups']
X_test, y_test = test['X'], test['y']
num_classes=len(classes)
input_shape= X_dev.shape[1:]

print(f'Dev: {X_dev.shape}  ({len(np.unique(groups_dev))} participants)')
print(f'Test: {X_test.shape}')
print(f'Classes: {num_classes}')
print(f'Random chance: {100/num_classes:.1f}%')

Dev: (16013, 30, 274)  (18 participants)
Test: (2835, 30, 274)
Classes: 50
Random chance: 2.0%


***Setting up LSTM framework***

This will be mostly the LSTM architecture that we had to design to be flexible with hyperparameter tuning, and especially the temporal jittering. The model is bi-directional unless specified to be false, because we want it to read each clip forward, then backward, since this is stated to be useful in ASL gesture recognition. All the 1st layer of the LSTM returns full sequences so the 2nd layer can process them, and with an addition to the last LSTM that returns the hidden state. The hidden state is pretty much a summary of the clip, which is controlled by the return_sequences variable, and with it being false, it collapses the 30-frame sequence into a single representation that the dense classifier head can map to a single layer since we are classifying the entirety of the clip and not each frame. 

In [3]:
# using bi directional because it is helpful to look at end pose then to beginning
def build_lstm(input_shape, num_classes=50,
               lstm_units=(128, 64), dropout=0.4, recurrent_dropout=0.0,
               dense_size=64, bidirectional=True, learning_rate=1e-3):

    def lstm_layer(units, return_sequences):
        layer = LSTM(units, return_sequences=return_sequences, recurrent_dropout=recurrent_dropout)
        return Bidirectional(layer) if bidirectional else layer

    layers = [Input(shape=input_shape)]
    # last lstm layer returns summary on whole clip
    for i, units in enumerate(lstm_units):
        is_last = (i == len(lstm_units) - 1)
        layers.append(lstm_layer(units, return_sequences=not is_last))
        layers.append(Dropout(dropout))

    layers += [
        Dense(dense_size, activation='relu'),
        BatchNormalization(),
        Dropout(dropout),
        Dense(num_classes, activation='softmax'),
    ]

    model = Sequential(layers)
    model.compile(optimizer=Adam(learning_rate=learning_rate), loss='categorical_crossentropy', metrics=['accuracy'])
    return model
build_lstm(input_shape=input_shape, num_classes=num_classes).summary()

2026-04-19 22:22:12.680651: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-19 22:22:13.312147: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1616] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 30968 MB memory:  -> device: 0, name: Tesla V100-SXM2-32GB, pci bus id: 0000:3a:00.0, compute capability: 7.0


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 bidirectional (Bidirectiona  (None, 30, 256)          412672    
 l)                                                              
                                                                 
 dropout (Dropout)           (None, 30, 256)           0         
                                                                 
 bidirectional_1 (Bidirectio  (None, 128)              164352    
 nal)                                                            
                                                                 
 dropout_1 (Dropout)         (None, 128)               0         
                                                                 
 dense (Dense)               (None, 64)                8256      
                                                                 
 batch_normalization (BatchN  (None, 64)               2

***Temporal Jitter Explanation***

With a high-end variance of 10-12%, we were trying to do reduce variance using temporal jitter. This was to try to counter the different speeds at which the participants would sign. So with this function, we could speed up or slow down a certain "clip" (the size of 30 always referred to the number of frames). 

In [4]:
def temporal_jitter(X, max_warp=0.15, seed=None):
    rng = np.random.default_rng(seed)
    N, T, F = X.shape
    out = np.empty_like(X)
    x_src = np.linspace(0, 1, T)
    for i in range(N):
        # random warp to help with hyperparameter tuning
        warp = rng.uniform(-max_warp, max_warp)
        x_new = x_src + warp * np.sin(np.pi * x_src) 
        x_new = np.clip(x_new, 0, 1)
        for j in range(F):
            out[i, :, j] = np.interp(x_new, x_src, X[i, :, j])
    return out

In [5]:
# Cross validation check
n_folds = 5
fold_accs = []

for fold in range(n_folds):
    print(f'\n=== Fold {fold+1}/{n_folds} ===')
    tr = folds[f'fold_{fold}_train']
    va = folds[f'fold_{fold}_val']
    X_tr, y_tr = X_dev[tr], y_dev[tr]
    X_va, y_va = X_dev[va], y_dev[va]
    y_tr_cat = to_categorical(y_tr, num_classes)
    y_va_cat = to_categorical(y_va, num_classes)

    model = build_lstm(input_shape = input_shape, num_classes=num_classes)
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-5),
    ]
    
    model.fit(X_tr, y_tr_cat, validation_data=(X_va, y_va_cat), epochs=80, batch_size=128, callbacks=callbacks, verbose=2)

    xx, acc = model.evaluate(X_va, y_va_cat, verbose=0)
    fold_accs.append(acc)
    print(f'Fold {fold+1} accuracy: {acc*100:.2f}%')

print(f'\nMean CV: {np.mean(fold_accs)*100:.2f}% ± {np.std(fold_accs)*100:.2f}%')


=== Fold 1/5 ===
Epoch 1/80


2026-04-19 22:22:20.353013: I tensorflow/stream_executor/cuda/cuda_dnn.cc:384] Loaded cuDNN version 8201


98/98 - 8s - loss: 4.3086 - accuracy: 0.0236 - val_loss: 3.8689 - val_accuracy: 0.0356 - lr: 0.0010 - 8s/epoch - 81ms/step
Epoch 2/80
98/98 - 1s - loss: 3.9659 - accuracy: 0.0384 - val_loss: 3.6872 - val_accuracy: 0.0661 - lr: 0.0010 - 1s/epoch - 11ms/step
Epoch 3/80
98/98 - 1s - loss: 3.7010 - accuracy: 0.0702 - val_loss: 3.4032 - val_accuracy: 0.1185 - lr: 0.0010 - 1s/epoch - 11ms/step
Epoch 4/80
98/98 - 1s - loss: 3.4319 - accuracy: 0.1053 - val_loss: 3.1676 - val_accuracy: 0.1504 - lr: 0.0010 - 1s/epoch - 11ms/step
Epoch 5/80
98/98 - 1s - loss: 3.1634 - accuracy: 0.1441 - val_loss: 2.9049 - val_accuracy: 0.1983 - lr: 0.0010 - 1s/epoch - 11ms/step
Epoch 6/80
98/98 - 1s - loss: 2.9659 - accuracy: 0.1802 - val_loss: 2.6983 - val_accuracy: 0.2544 - lr: 0.0010 - 1s/epoch - 11ms/step
Epoch 7/80
98/98 - 1s - loss: 2.8047 - accuracy: 0.2147 - val_loss: 2.6340 - val_accuracy: 0.2587 - lr: 0.0010 - 1s/epoch - 11ms/step
Epoch 8/80
98/98 - 1s - loss: 2.6744 - accuracy: 0.2455 - val_loss: 2.536

In [6]:
# hyper paramter tuning
param_grid = [
    # Baseline
    {'lstm_units': (128, 64), 'dropout': 0.4, 'bidirectional': True},
    # Smaller lstm
    {'lstm_units': (64, 32),  'dropout': 0.4, 'bidirectional': True},
    # Higher dropout value
    {'lstm_units': (128, 64), 'dropout': 0.5, 'bidirectional': True},
    # change to 1 direction
    {'lstm_units': (128, 64), 'dropout': 0.4, 'bidirectional': False},
    # Decrease learning rate
    {'lstm_units': (128, 64), 'dropout': 0.4, 'bidirectional': True, 'learning_rate': 5e-4},
    # Decrease learning rate
    {'lstm_units': (128, 64), 'dropout': 0.4, 'bidirectional': True, 'learning_rate': 2e-3},
    # 3 Layer LSTM
    {'lstm_units': (64, 64, 64), 'dropout': 0.4, 'bidirectional': True, 'learning_rate': 1e-3},
]

tuning_results = []

for params in param_grid:
    print(f'\n\nConfig: {params}')
    accs = []
    for fold in range(n_folds):
        tr = folds[f'fold_{fold}_train']
        va = folds[f'fold_{fold}_val']
        y_tr_cat = to_categorical(y_dev[tr], num_classes)
        y_va_cat = to_categorical(y_dev[va], num_classes)

        model = build_lstm(input_shape = input_shape, num_classes=num_classes, **params)
        model.fit(X_dev[tr], y_tr_cat, validation_data=(X_dev[va], y_va_cat), epochs=60, batch_size=128,
                  # using callbacks to stop overfitting from happening (just looks at when val loss stops decreasing after 8 epochs)
                  callbacks=[
                      EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True),
                      ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-5),
                  ],
                  verbose=0)
        q, acc = model.evaluate(X_dev[va], y_va_cat, verbose=0)
        accs.append(acc)
        print(f'Fold {fold+1}: {acc*100:.2f}%')
    mean_acc = np.mean(accs)
    std_acc  = np.std(accs)
    tuning_results.append((params, mean_acc, std_acc))
    print(f'Mean: {mean_acc*100:.2f}% ± {std_acc*100:.2f}%')

best_params, best_acc, xxx = max(tuning_results, key=lambda r: r[1])



Config: {'lstm_units': (128, 64), 'dropout': 0.4, 'bidirectional': True}
Fold 1: 57.24%
Fold 2: 58.80%
Fold 3: 55.64%
Fold 4: 59.63%
Fold 5: 57.59%
Mean: 57.78% ± 1.37%


Config: {'lstm_units': (64, 32), 'dropout': 0.4, 'bidirectional': True}
Fold 1: 54.42%
Fold 2: 54.63%
Fold 3: 54.67%
Fold 4: 54.53%
Fold 5: 54.06%
Mean: 54.46% ± 0.22%


Config: {'lstm_units': (128, 64), 'dropout': 0.5, 'bidirectional': True}
Fold 1: 51.28%
Fold 2: 57.39%
Fold 3: 53.57%
Fold 4: 57.05%
Fold 5: 55.52%
Mean: 54.96% ± 2.28%


Config: {'lstm_units': (128, 64), 'dropout': 0.4, 'bidirectional': False}
Fold 1: 52.91%
Fold 2: 55.59%
Fold 3: 51.85%
Fold 4: 54.75%
Fold 5: 52.30%
Mean: 53.48% ± 1.44%


Config: {'lstm_units': (128, 64), 'dropout': 0.4, 'bidirectional': True, 'learning_rate': 0.0005}
Fold 1: 54.96%
Fold 2: 56.65%
Fold 3: 56.92%
Fold 4: 54.99%
Fold 5: 55.01%
Mean: 55.70% ± 0.89%


Config: {'lstm_units': (128, 64), 'dropout': 0.4, 'bidirectional': True, 'learning_rate': 0.002}
Fold 1: 56.04%
Fold 2

In [ ]:
# final model with best params — trained on full dev set
y_dev_cat  = to_categorical(y_dev,  num_classes)
y_test_cat = to_categorical(y_test, num_classes)

final_model = build_lstm(input_shape = input_shape, num_classes=num_classes, **best_params)
final_model.fit(X_dev, y_dev_cat,
    epochs=80, batch_size=128,
    # ReduceLROnPlateau monitors training loss and halves LR if it stagnates for 4 epochs
    callbacks=[
        ReduceLROnPlateau(monitor='loss', factor=0.5, patience=4, min_lr=1e-5),
    ],
    verbose=2
)

xx, test_acc = final_model.evaluate(X_test, y_test_cat, verbose=0)
print(f'\nHeld-out test accuracy: {test_acc*100:.2f}%')
print(f'Random chance: {100/num_classes:.1f}%')

Epoch 1/80
126/126 - 8s - loss: 4.2109 - accuracy: 0.0257 - lr: 0.0010 - 8s/epoch - 62ms/step
Epoch 2/80
126/126 - 1s - loss: 3.7965 - accuracy: 0.0560 - lr: 0.0010 - 1s/epoch - 11ms/step
Epoch 3/80
126/126 - 1s - loss: 3.5112 - accuracy: 0.0919 - lr: 0.0010 - 1s/epoch - 11ms/step
Epoch 4/80
126/126 - 1s - loss: 3.2122 - accuracy: 0.1403 - lr: 0.0010 - 1s/epoch - 12ms/step
Epoch 5/80
126/126 - 2s - loss: 2.9593 - accuracy: 0.1877 - lr: 0.0010 - 2s/epoch - 12ms/step
Epoch 6/80
126/126 - 1s - loss: 2.7891 - accuracy: 0.2219 - lr: 0.0010 - 1s/epoch - 12ms/step
Epoch 7/80
126/126 - 1s - loss: 2.6415 - accuracy: 0.2452 - lr: 0.0010 - 1s/epoch - 11ms/step
Epoch 8/80
126/126 - 1s - loss: 2.5185 - accuracy: 0.2806 - lr: 0.0010 - 1s/epoch - 11ms/step
Epoch 9/80
126/126 - 1s - loss: 2.4436 - accuracy: 0.3018 - lr: 0.0010 - 1s/epoch - 11ms/step
Epoch 10/80
126/126 - 1s - loss: 2.3336 - accuracy: 0.3234 - lr: 0.0010 - 1s/epoch - 12ms/step
Epoch 11/80
126/126 - 1s - loss: 2.2676 - accuracy: 0.3458 

In [ ]:
# per class accuracy
y_pred = final_model.predict(X_test, verbose=0).argmax(axis=1)
print(classification_report(y_test, y_pred, target_names=classes, zero_division=0))

In [ ]:
# confusion matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(16, 14))
sns.heatmap(cm, annot=True, fmt ='d', xticklabels=classes, yticklabels=classes, cmap='Blues', square=True, cbar_kws={'label': 'Count'})
plt.xlabel('Predict')
plt.ylabel('True')
plt.title('LSTM Confusion Matrix')
plt.xticks(rotation=90)
plt.show()

cm_no_diag = cm.copy()
np.fill_diagonal(cm_no_diag, 0)
print('\nTop 10 confused pairs:')
for idx in np.argsort(cm_no_diag.ravel())[-10:][::-1]:
    i, j = np.unravel_index(idx, cm_no_diag.shape)
    if cm_no_diag[i, j] > 0:
        print(f'{classes[i]} confused as {classes[j]} : {cm_no_diag[i, j]} times')